# Notebook 1 — Exploration des Données
**Pipeline Big Data de Monitoring de la Désinformation en Temps Réel**  
KOMOSSI Sosso — Master 2 IBDIA, UCAO-UUT 2025-2026

Ce notebook explore les 4 datasets utilisés :
- **WELFake** (Zenodo) — 72 134 articles
- **ISOT** (University of Victoria) — 44 898 articles
- **LIAR** (UCSB) — 12 836 statements
- **FakeNewsNet** (GitHub KaiDMML) — gossipcop + politifact


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys
sys.path.insert(0, '..')

BASE = '../data'
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')
print('Librairies chargées avec succès')

## 1. Chargement des données prétraitées

In [ ]:
train = pd.read_csv(f'{BASE}/processed/train/train.csv')
val   = pd.read_csv(f'{BASE}/processed/val/val.csv')
test  = pd.read_csv(f'{BASE}/processed/test/test.csv')

print(f'Train : {len(train):>7,} exemples')
print(f'Val   : {len(val):>7,} exemples')
print(f'Test  : {len(test):>7,} exemples')
print(f'TOTAL : {len(train)+len(val)+len(test):>7,} exemples')
train.head(3)

## 2. Distribution des classes (fake / réel)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (df, title) in zip(axes, [(train,'Train'),(val,'Validation'),(test,'Test')]):
    counts = df['label'].value_counts()
    ax.pie(counts, labels=['Réel (0)','Fake (1)'], autopct='%1.1f%%',
           colors=['#2ecc71','#e74c3c'], startangle=90)
    ax.set_title(f'{title} ({len(df):,} articles)')
plt.suptitle('Distribution des classes par split', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{BASE}/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Distribution par source (dataset)

In [ ]:
corpus = pd.concat([train, val, test], ignore_index=True)
src_counts = corpus.groupby(['source','label']).size().unstack(fill_value=0)
src_counts.columns = ['Réel', 'Fake']

ax = src_counts.plot(kind='bar', color=['#2ecc71','#e74c3c'], figsize=(10,5))
ax.set_title("Nombre d'articles par dataset et par classe", fontsize=13)
ax.set_xlabel('Dataset')
ax.set_ylabel("Nombre d'articles")
ax.legend(title='Classe')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{BASE}/processed/source_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(src_counts)

## 4. Longueur des textes (titre / corps)

In [ ]:
train['title_len'] = train['title'].str.len()
train['body_len']  = train['body'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, label in zip(axes, ['title_len','body_len'], ['Titre','Corps']):
    for lbl, color in [(0,'#2ecc71'),(1,'#e74c3c')]:
        ax.hist(train[train['label']==lbl][col].clip(0,500), bins=50,
                alpha=0.6, color=color, label='Réel' if lbl==0 else 'Fake')
    ax.set_title(f'Longueur du {label} (caractères)')
    ax.set_xlabel('Longueur')
    ax.legend()
plt.tight_layout()
plt.show()

print('Statistiques — Longueur du titre :')
print(train.groupby('label')['title_len'].describe().round(1))

## 5. Résumé statistique

In [ ]:
print('=== Résumé du corpus ===')
print(f'Total examples  : {len(corpus):,}')
print(f'Fake rate       : {corpus["label"].mean()*100:.1f}%')
print(f'Sources         : {corpus["source"].unique().tolist()}')
print(f'Titre moyen     : {corpus["title"].str.len().mean():.0f} caractères')
print(f'Corps moyen     : {corpus["body"].fillna("").str.len().mean():.0f} caractères')
print()
print('Répartition par source et classe :')
print(corpus.groupby(['source','label']).size().to_string())